# Salis Lab Promoter Calculator — Example

![Salis Lab Promoter Calculator — Example](https://proto-bio.github.io/proto-assets/images/tool/promoter_calculator/hero.png)

This notebook demonstrates `run_promoter_calculator`, which predicts sigma70 promoter strength on DNA sequences in *E. coli*. The calculator scans both strands for canonical sigma70 elements (UP, -35 hexamer, spacer, -10 hexamer, discriminator) and returns the binding free energy `dG_total` (kcal/mol) and predicted transcription initiation rate `Tx_rate` (au) for each candidate promoter.

Reference: LaFleur et al., *Nature Communications* 2022, [DOI 10.1038/s41467-022-32829-5](https://doi.org/10.1038/s41467-022-32829-5).

In [1]:
from proto_tools.tools.gene_annotation.promoter_calculator import (
    PromoterCalculatorConfig,
    PromoterCalculatorInput,
    run_promoter_calculator,
)

## Input API Reference

`PromoterCalculatorInput` fields:

| Field | Type | Default | Description |
|---|---|---|---|
| `sequences` | `list[str]` | *required* | DNA sequences to scan for sigma70 promoters |

## Config API Reference

`PromoterCalculatorConfig` fields:

| Field | Type | Default | Description |
|---|---|---|---|
| `threads` | `int` | `1` | CPU threads for parallel scan |
| `circular` | `bool` | `False` | Treat sequences as circular DNA (for plasmids) |

## Output API Reference

`PromoterCalculatorOutput.results` is a `list[PromoterCalculatorSequenceResult]`, one per input sequence. Each `PromoterCalculatorSequenceResult` carries:

- `sequence_id: str` — the input sequence identifier
- `predictions: list[PromoterPrediction]` — every predicted promoter on both strands

Each `PromoterPrediction` has the following fields:

| Field | Type | Description |
|---|---|---|
| `tss_name` | `str` | Label like `Fwd123` / `Rev456` |
| `tss` | `int` | Transcription start site position |
| `strand` | `str` | `+` or `-` |
| `dG_total` | `float` | Binding free energy (kcal/mol); more negative = stronger |
| `Tx_rate` | `float` | Predicted transcription initiation rate (au); higher = stronger |
| `promoter_sequence` | `str` | DNA sequence spanning the predicted promoter |
| `length` | `int` | Length of the promoter sequence |
| `UP_position` | `[int, int]` | UP element bounds |
| `hex35_position` | `[int, int]` | -35 hexamer bounds |
| `spacer_position` | `[int, int]` | spacer bounds |
| `hex10_position` | `[int, int]` | -10 hexamer bounds |
| `disc_position` | `[int, int]` | discriminator bounds |

In [2]:
# lacUV5 promoter (well-characterized E. coli sigma70 promoter)
lac_uv5 = "AAAATTGTGAGCGGATAACAATTTCACACAGGAAACAGCTATGACC"
sequence = "A" * 20 + lac_uv5 + "A" * 20

inputs = PromoterCalculatorInput(sequences=[sequence])
config = PromoterCalculatorConfig(threads=1)
result = run_promoter_calculator(inputs, config)

print(f"Predicted promoters: {result.results[0].num_promoters}")
for pred in result.results[0].predictions[:3]:
    print(f"  {pred.tss_name}  strand={pred.strand}  "
          f"dG={pred.dG_total:.2f} kcal/mol  Tx_rate={pred.Tx_rate:.0f}")

Running run_promoter_calculator [00:00]

Predicted promoters: 5
  Rev61  strand=-  dG=-1.83 kcal/mol  Tx_rate=838
  Rev66  strand=-  dG=-1.63 kcal/mol  Tx_rate=607
  Fwd66  strand=+  dG=-1.48 kcal/mol  Tx_rate=475


In [3]:
# Compare two promoter variants in batch, with circular DNA handling
variants = {
    "J23119_strong": "TTGACAGCTAGCTCAGTCCTAGGTATAATGCTAGC",
    "J23113_weak":   "CTGATGGCTAGCTCAGTCCTAGGGATTATGCTAGC",
}
inputs = PromoterCalculatorInput(
    sequences=["A" * 30 + s + "A" * 30 for s in variants.values()],
)
config = PromoterCalculatorConfig(threads=4, circular=False)
result = run_promoter_calculator(inputs, config)

for seq_result in result.results:
    best = max(seq_result.predictions, key=lambda p: p.Tx_rate, default=None)
    if best is None:
        print(f"{seq_result.sequence_id}: no promoter detected")
    else:
        print(f"{seq_result.sequence_id}: best Tx_rate={best.Tx_rate:.0f} "
              f"(dG={best.dG_total:.2f}, strand={best.strand})")

Running run_promoter_calculator [00:00]

J23119_strong: best Tx_rate=83433 (dG=-4.64, strand=+)
J23113_weak: best Tx_rate=3522 (dG=-2.71, strand=+)


In [4]:
from pathlib import Path

out_dir = Path("./promoter_results")
out_dir.mkdir(exist_ok=True)
result.export(name="promoters", export_path=str(out_dir), file_format="csv")
result.export(name="promoters", export_path=str(out_dir), file_format="json")

print("Wrote:", sorted(p.name for p in out_dir.iterdir()))

Wrote: ['promoters.csv', 'promoters.json']
